# Building a Modular Semantic Search Pipeline (RAG)

This notebook implements a basic **Retrieval-Augmented Generation (RAG)** retrieval pipeline using Object-Oriented Programming (OOP). 

### How Semantic Search Works:
1. **Chunking**: Large documents are broken down into smaller pieces so they fit within an AI model's context window.
2. **Embedding**: Text chunks are converted into dense mathematical vectors (lists of numbers) that capture semantic meaning.
3. **Cosine Similarity**: User queries are vectorized, and we calculate the mathematical angle between the query vector and our chunk vectors to find the closest match.

Let's start by installing our core AI library and importing the required utilities.

In [1]:
# %% [code]
# Install sentence-transformers if you haven't already: !pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
import numpy as np
import torch
from typing import List, Dict, Any, Tuple

# Mock database simulating documents in a knowledge base
DOCS = [
    {
        "doc_id": "doc-ai",
        "title": "Intro to Large Language Models",
        "text": (
            "Large language models use transformer architectures to process text. "
            "They represent meaning as dense vectors called embeddings. "
            "Attention mechanisms help models focus on relevant parts of input. "
            "RAG systems retrieve relevant chunks and feed them back to the model. "
            "Good chunking improves retrieval quality and final answer accuracy."
        ),
    },
    {
        "doc_id": "doc-analytics",
        "title": "Product Analytics Guide",
        "text": (
            "Product analytics tracks user behavior to improve features. "
            "Metrics include retention, engagement, and conversion rate. "
            "Event data is collected from apps and websites and stored in warehouses. "
            "Dashboards help teams test hypotheses and prioritize experiments. "
            "Clear instrumentation is essential for trustworthy insights."
        ),
    },
]
print(f"Loaded {len(DOCS)} source documents.")

c:\Projects\Mission-Ready_AI_engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 2 source documents.


## Component 1: The Document Chunker

Embedding models have text length limitations. If we feed a whole book to a model, the specific details get washed out. 

The `DocumentChunker` class uses a **Sliding Window** strategy:
* **`max_words`**: The absolute maximum size of a text chunk.
* **`overlap`**: The number of words shared between consecutive chunks. This prevents vital information from being split in half and losing its context at the borders.

In [3]:
# %% [code]
class DocumentChunker:
    """Handles splitting large text structures into overlapping, manageable segments."""
    def __init__(self, max_words: int = 30, overlap: int = 8):
        self.max_words = max_words
        self.overlap = overlap

    def chunk_document(self, doc: Dict[str, str]) -> List[Dict[str, Any]]:
        text = doc["text"]
        doc_id = doc["doc_id"]
        title = doc["title"]
        
        words = text.split()
        chunks = []
        start = 0
        chunk_id = 0

        while start < len(words):
            end = min(start + self.max_words, len(words))
            chunk_words = words[start:end]
            chunk_text = " ".join(chunk_words)

            chunks.append({
                "chunk_id": f"{doc_id}-{chunk_id}",
                "doc_id": doc_id,
                "title": title,
                "text": chunk_text,
                "span": (start, end),
            })

            if end == len(words):
                break
            start = end - self.overlap
            chunk_id += 1

        return chunks

    def chunk_corpus(self, docs: List[Dict[str, str]]) -> List[Dict[str, Any]]:
        all_chunks = []
        for doc in docs:
            all_chunks.extend(self.chunk_document(doc))
        return all_chunks

# Test the chunker instantly to inspect its behavior
chunker = DocumentChunker(max_words=20, overlap=5)
sample_chunks = chunker.chunk_corpus(DOCS)
print(f"Generated {len(sample_chunks)} chunks from source documents.")
print(f"Sample Chunk 1: {sample_chunks[0]['text']}")

Generated 6 chunks from source documents.
Sample Chunk 1: Large language models use transformer architectures to process text. They represent meaning as dense vectors called embeddings. Attention mechanisms help


## Component 2: The Semantic Search Engine

This class encapsulates our data model state. It initializes a lightweight, fast text embedding model (`all-MiniLM-L6-v2`) which maps sentences into a 384-dimensional dense vector space.

### Key Class Methods:
1. **`add_documents()`**: Loops through text chunks, translates them into math vectors via `model.encode()`, and saves them to memory (`self.embeddings`).
2. **`search()`**: Converts a natural language string query into a single vector, performs a mathematical vector dot-product matrix multiplication against all cached documents, and pulls the highest scoring results.

In [4]:
# %% [code]
class SemanticSearchEngine:
    """Manages text embedding generation, internal index memory, and cosine proximity searches."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # Load the sentence transformer model into memory
        self.model = SentenceTransformer(model_name)
        self.chunks: List[Dict[str, Any]] = []
        self.embeddings: np.ndarray = np.empty((0,))

    def add_documents(self, chunks: List[Dict[str, Any]]):
        """Generates vectors for raw text chunks and appends them to the index state."""
        if not chunks:
            return
            
        self.chunks.extend(chunks)
        texts = [c["text"] for c in chunks]
        
        # normalize_embeddings=True calculates L2 normalized vectors 
        # This allows simple dot-products to accurately represent Cosine Similarity
        new_embeddings = self.model.encode(
            texts, 
            convert_to_tensor=True, 
            normalize_embeddings=True
        )
        
        if self.embeddings.size == 0:
            self.embeddings = new_embeddings
        else:
            self.embeddings = torch.cat([self.embeddings, new_embeddings], dim=0)

    def search(self, query: str, top_k: int = 3) -> List[Dict[str, Any]]:
        """Executes a semantic match query over the stored vector index."""
        if self.embeddings.size == 0:
            raise ValueError("Vector index is completely empty. Please run add_documents first.")

        # Vectorize incoming query text
        q_emb = self.model.encode([query], convert_to_tensor=True, normalize_embeddings=True)
        
        # util.semantic_search handles high-performance cosine matching
        hits = util.semantic_search(q_emb, self.embeddings, top_k=top_k)[0]

        results = []
        for h in hits:
            ch = self.chunks[h["corpus_id"]]
            results.append({
                "score": float(h["score"]),
                "chunk_id": ch["chunk_id"],
                "title": ch["title"],
                "text": ch["text"],
                "span": ch["span"]
            })
        return results

## Running and Verifying the Search Pipeline

Now we piece our objects together. We initialize our classes, build our chunk index, and test the search capability against conceptual semantic queries. 

Notice how the engine will successfully retrieve accurate snippets even if the query uses slightly different vocabulary than the document text.

In [5]:
# %% [code]
# 1. Pipeline Instantiation
engine = SemanticSearchEngine()
document_splitter = DocumentChunker(max_words=25, overlap=6)

# 2. Document preparation
processed_chunks = document_splitter.chunk_corpus(DOCS)
engine.add_documents(processed_chunks)

# 3. Test Query Evaluation
test_query = "What data metrics track user activity?"
search_results = engine.search(test_query, top_k=2)

print(f"Query Run: '{test_query}'\n")
for idx, match in enumerate(search_results, 1):
    print(f"Match [{idx}] Confidence Score: {match['score']:.4f}")
    print(f"Source: {match['title']} ({match['chunk_id']})")
    print(f"Content: '{match['text']}'\n" + "-"*40)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 21067.65it/s]


Query Run: 'What data metrics track user activity?'

Match [1] Confidence Score: 0.6550
Source: Product Analytics Guide (doc-analytics-0)
Content: 'Product analytics tracks user behavior to improve features. Metrics include retention, engagement, and conversion rate. Event data is collected from apps and websites and stored'
----------------------------------------
Match [2] Confidence Score: 0.4364
Source: Product Analytics Guide (doc-analytics-1)
Content: 'from apps and websites and stored in warehouses. Dashboards help teams test hypotheses and prioritize experiments. Clear instrumentation is essential for trustworthy insights.'
----------------------------------------
